In [ ]:
import polars as pl

folder_paths = [
    "/lakehouse/default/Files/PerfScenarios/logs/db_cm_10_01",
    "/lakehouse/default/Files/PerfScenarios/logs/db_cm_10_02",
    ...
    "/lakehouse/default/Files/PerfScenarios/logs/db_dq_10_01",
    "/lakehouse/default/Files/PerfScenarios/logs/db_dq_10_02",
    ...
    "/lakehouse/default/Files/PerfScenarios/logs/fab_dl_100_01",
    "/lakehouse/default/Files/PerfScenarios/logs/fab_dl_100_02",
    ...
    "/lakehouse/default/Files/PerfScenarios/logs/fab_mirror_100_02",
    "/lakehouse/default/Files/PerfScenarios/logs/fab_mirror_100_03"
]                         # add all folder paths for the load tests, follow the naming standard if you seek to use this script as is to analyze the results, otherwise change the logic in the next cell 

# Consolodate CSV log files of results into single dataframe
df_list = []
for folder_path in folder_paths:
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            polars_df = pl.read_csv(file_path)


            polars_df = (polars_df
                .with_columns([
                     pl.col("start_time_dt").str.to_datetime(time_unit="ms").dt.truncate("1s").alias("start_time_s").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            polars_df = (polars_df
                .with_columns([
                    pl.col("start_time_dt").str.to_datetime(time_unit="ms").alias("start_time_dt").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            df_list.append(polars_df)

test_df = pl.concat(df_list)
display(test_df)

In [ ]:
parsed_df = test_df.with_columns([
    pl.col("loadtest_id").str.extract(r"^(.*)_(\d+)_(\d+)-\d+-\d+$", 1).alias("Pattern"),
    pl.col("loadtest_id").str.extract(r"^(.*)_(\d+)_(\d+)-\d+-\d+$", 2).alias("SF"),
    pl.col("loadtest_id").str.extract(r"^(.*)_(\d+)_(\d+)-\d+-\d+$", 3).cast(pl.Int64).cast(pl.Utf8).alias("Run"),
])


renamed_df = parsed_df.with_columns(
    pl.col("Pattern").replace({
        "db_cm": "DB_Composite Model",
        "db_dq": "Direct_Query",
        "fab_dl": "Direct_Lake",
        "fab_mirror": "DL_Mirror",
    }).alias("Pattern")
)

# make duration miliseconds
clean_df=renamed_df.with_columns(
    (pl.col("duration") * 1000).alias("duration"))

display(clean_df)

In [ ]:
import notebookutils

table_path = notebookutils.lakehouse.get(name="").properties["abfsPath"] + "/Tables/dbo/loadtest_raw"   # populate name of metrics analysis lakehouse to store the results
clean_df.write_delta(table_path, mode="overwrite")

In [ ]:
combined_df = clean_df.group_by(["SF", "Pattern", "query_number","Run","visual_name"]).agg(pl.col("duration").quantile(0.5).alias("p50")).sort(["SF", "Pattern", "query_number","Run"])
display(combined_df)

In [ ]:
table_path = notebookutils.lakehouse.get(name="").properties["abfsPath"] + "/Tables/dbo/loadtest_p50_per_run"  # populate name of metrics analysis lakehouse to store the results

combined_df.write_delta(table_path, mode="overwrite")

In [ ]:
result_df = combined_df.group_by(["SF", "Pattern", "query_number", "visual_name"]).agg(
    pl.col("p50").mean().alias("p50_avg"),
    pl.col("p50").count().alias("run_count"),
).sort(["SF", "Pattern", "query_number"])

display(result_df)

In [ ]:
table_path = notebookutils.lakehouse.get(name="").properties["abfsPath"] + "/Tables/dbo/loadtest_p50_average"  # populate name of metrics analysis lakehouse to store the results

result_df.write_delta(table_path, mode="overwrite")